# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [66]:
# import
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import base64
from io import BytesIO
from PIL import Image
import json

In [67]:
# initialization
#add model choose
load_dotenv(override=True)
openai = OpenAI()
MODEL = 'gpt-5'

In [68]:
# system prompt
system_message = """
        You are a helpful and friendly technical expert and teacher. When a student asks a question, answer it clearly and in detail, using practical examples.
        You should use the image generation tool to create an illustration that helps to visualize the concept.
        First, answer the student's question directly. Then, explain the generated image in the context of your explanation. Tell the student how the image relates to the concept, rather than giving a dry, technical description of the image's appearance or the prompt used to create it. Reply in Markdown, make your respond consistent like a statement.
                """

In [69]:
def generate_image(image_prompt):
    image_response = openai.images.generate(
            model="dall-e-3",
            prompt=image_prompt,
            size="1024x1024",
            n=1,
            response_format="b64_json",
        )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

In [70]:
def generate_voice_respond(message):
    response = openai.audio.speech.create(
      model="gpt-4o-mini-tts",
      voice="onyx",
      input=message
    )
    return response.content

In [71]:
def transcribe(audio_path):
    if audio_path is None:
        return ""
    with open(audio_path, "rb") as audio_file:
        transcript = openai.audio.transcriptions.create(
            model="whisper-1",
            file=audio_file
        )
    return transcript.text

In [72]:
image_function = {
    "name": "generate_image",
    "description": "Enter a prompt to generate image.",
    "parameters": {
        "type": "object",
        "properties": {
            "image_prompt": {
                "type": "string",
                "description": "The explanation of image",
            },
        },
        "required": ["image_prompt"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": image_function}]

In [73]:
def handle_tool_calls(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == 'generate_image':
        arguments = json.loads(tool_call.function.arguments)
        image_prompt = arguments.get('image_prompt')
        image = generate_image(image_prompt)
        response = {
            "role": "tool",
            "content": "Image generated successfully.",
            "tool_call_id": tool_call.id,
            "image": image
        }
        return response

In [74]:
def chat(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools = tools)
    image = None

    while response.choices[0].finish_reason=="tool_calls":
         message = response.choices[0].message
         tool_response = handle_tool_calls(message)
         image = tool_response.pop("image")
         messages.append(message)
         messages.append(tool_response)
         response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]
    voice_respond = generate_voice_respond(reply)

    return history, voice_respond, image


In [75]:
# maybe add radio button
def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height = 500, type='messages')
        image_output = gr.Image(height = 500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        text_input = gr.Textbox(placeholder='Enter your message ...' , label='Chat with our technical AI Assistant:')
    with gr.Row():
        audio_input =  gr.Audio(label='Speak to your ai assistant.', type="filepath")
# audio input -> text input
    audio_input.change(transcribe, inputs=[audio_input], outputs=[text_input])
    text_input.submit(put_message_in_chatbot, inputs=[text_input, chatbot], outputs=[text_input, chatbot]).then(
            chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
    )

ui.launch(inbrowser=True, auth=("eryk", "banana"))

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.
